# ResQ Mega Training (YOLOv8 Detection)
This notebook trains your custom person/wound dataset alongside the COCO dataset to prevent **catastrophic forgetting**. 

Ensure you have uploaded `combined_dataset.zip` to your Google Drive before running.

In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Extract Custom Dataset
Assuming `combined_dataset.zip` is sitting in your Google Drive root (`MyDrive`).

In [ ]:
import os
import shutil

drive_zip_path = "/content/drive/MyDrive/combined_dataset.zip"
local_zip_path = "/content/combined_dataset.zip"
local_extract_dir = "/content/custom_dataset"

if os.path.exists(drive_zip_path):
    print(f"File found! Size: {os.path.getsize(drive_zip_path) / (1024*1024):.2f} MB")
    shutil.copy(drive_zip_path, local_zip_path)
    # Added -o to ALWAYS overwrite so it doesn't get stuck asking [y]es or [No]
    !unzip -o -q {local_zip_path} -d {local_extract_dir}
    print("Successfully extracted custom dataset.")
else:
    print(f"ERROR: Could not find {drive_zip_path}. Please make sure you uploaded combined_dataset.zip to your Google Drive.")

## 2. Download COCO Regularization Dataset

In [ ]:
from ultralytics.utils.downloads import download

print("Downloading COCO128 dataset...")
download("https://ultralytics.com/assets/coco128.zip", dir="/content")

## 3. Merge Datasets via YAML

In [ ]:
import yaml
import os
import glob
import shutil

print("--- DIAGNOSTICS & FORMATTING ---")

# 1. DELETE ANY OLD WINDOWS CACHE FILES
print("Cleaning old cache files...")
for cache_file in glob.glob('/content/custom_dataset/**/*.cache', recursive=True):
    os.remove(cache_file)
for cache_file in glob.glob('/content/datasets/**/*.cache', recursive=True):
    os.remove(cache_file)

# 2. FIX CUSTOM DATASET LABELS (POSE -> OBJ DETECTION)
# The issue is your dataset has pose keypoints (55+ numbers per line).
# The standard YOLOv8 object detector only expects 5 numbers (class, x, y, width, height).
print("Converting pose labels to standard bounding box labels...")
fixed_files = 0
for label_path in glob.glob('/content/custom_dataset/combined_dataset/labels/**/*.txt', recursive=True):
    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 5:
            # Keep only the first 5 elements: class, x_center, y_center, width, height
            new_line = " ".join(parts[:5]) + "\n"
            new_lines.append(new_line)
            
    with open(label_path, 'w') as f:
        f.writelines(new_lines)
    fixed_files += 1
print(f"Format fixed for {fixed_files} label files.")

# 3. LOCATE COCO128
coco_base = '/root/datasets/coco128' if os.path.exists('/root/datasets/coco128') else (
    '/content/datasets/coco128' if os.path.exists('/content/datasets/coco128') else '/content/coco128'
)

# 4. FILTER COCO LABELS TO ISOLATE "PERSON"
print("\n--- FILTERING COCO LABELS ---")
coco_labels_dir = f'{coco_base}/labels/train2017'
filtered_count = 0
if os.path.exists(coco_labels_dir):
    for label_file in glob.glob(f'{coco_labels_dir}/*.txt'):
        with open(label_file, 'r') as f:
            lines = f.readlines()
        
        # Keep ONLY class '0' (person)
        new_lines = [line for line in lines if line.startswith('0 ')]
        
        with open(label_file, 'w') as f:
            f.writelines(new_lines)
        if new_lines:
            filtered_count += 1
print(f"Kept 'person' labels in {filtered_count} COCO128 images. Discarded other classes.")

# 5. VERIFY FILE COUNTS
custom_train_imgs = len(glob.glob('/content/custom_dataset/combined_dataset/images/train/*'))
custom_val_imgs = len(glob.glob('/content/custom_dataset/combined_dataset/images/val/*'))
coco_imgs = len(glob.glob(f'{coco_base}/images/train2017/*'))

print("\n--- DATASET SIZES ---")
print(f"Custom Train Images: {custom_train_imgs}")
print(f"Custom Val Images: {custom_val_imgs}")
print(f"COCO Train Images: {coco_imgs}")

# 6. WRITE THE YAML USING ABSOLUTE PATHS
mega_yaml = {
    'path': '/content', # Base directory
    'train': [
        '/content/custom_dataset/combined_dataset/images/train',
        f'{coco_base}/images/train2017'
    ],
    'val': [
        '/content/custom_dataset/combined_dataset/images/val'
    ], 
    'names': {
        0: 'person',
        1: 'wound'
    }
}

with open('/content/mega_dataset.yaml', 'w') as f:
    yaml.dump(mega_yaml, f, sort_keys=False)

print("\nCreated mega_dataset.yaml successfully.")

## 4. Train the Standard YOLOv8 Object Detection Model

In [ ]:
from ultralytics import YOLO

# IMPORTANT: Using standard yolov8m.pt (Object Detection), NOT the pose model!
model = YOLO("yolov8m.pt")

results = model.train(
    data="/content/mega_dataset.yaml",
    epochs=50,
    imgsz=640,       # 640 to prevent CUDA Out of Memory errors
    batch=16,
    device=0,        # Use GPU 0
    project="ResQ_Final",
    name="ultimate_model"
)

## 5. Save the Weights to Drive

In [ ]:
from google.colab import files

# To download the file directly to your computer's Downloads folder
best_weight_path = "/content/runs/detect/ResQ_Final/ultimate_model6/weights/best.pt"
print(f"Downloading {best_weight_path}...")
files.download(best_weight_path)